In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# ---------------- SETTINGS ----------------

MAIN_URL = "https://www.spinny.com/used-cars-in-delhi-ncr/s/"

# ---------------- SETUP (DISABLE IMAGES) ----------------

options = Options()
options.add_argument("--start-maximized")

# Disable images
options.set_preference("permissions.default.image", 2)
options.set_preference("dom.ipc.plugins.enabled.libflashplayer.so", "false")

driver = webdriver.Firefox(options=options)
wait = WebDriverWait(driver, 35)

driver.get(MAIN_URL)

# ---------------- CLOSE POPUP ----------------

try:
    popup = WebDriverWait(driver, 6).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(.,'later') or contains(.,'Later')]"))
    )
    popup.click()
    print("Popup closed")
except:
    print("No popup")

# ---------------- LOAD ALL CARS (FASTER NOW) ----------------

def load_all_cars():

    last_count = 0
    same_count = 0

    while True:

        cards = driver.find_elements(By.CSS_SELECTOR, "div.CarListingDesktop__carListingCarWrapper")
        current_count = len(cards)

        print("Cars loaded:", current_count)

        if current_count == last_count:
            same_count += 1
        else:
            same_count = 0

        if same_count >= 3:
            print("All cars loaded.")
            break

        last_count = current_count

        driver.execute_script("window.scrollBy(0, 900);")

        # Faster because images disabled
        time.sleep(3)

load_all_cars()

cards = driver.find_elements(By.CSS_SELECTOR, "div.CarListingDesktop__carListingCarWrapper")
print("Final total cars:", len(cards))

master_data = []

# ---------------- LOOP THROUGH CARS ----------------

for index in range(len(cards)):

    print(f"\nProcessing Car {index+1}")

    try:
        cards = driver.find_elements(By.CSS_SELECTOR, "div.CarListingDesktop__carListingCarWrapper")
        card = cards[index]

        car_name = card.find_element(By.CSS_SELECTOR, "span.ListingBrandModelDetail__make").text
        car_link = card.find_element(By.TAG_NAME, "a").get_attribute("href")

        print("Opening:", car_name)

        driver.execute_script("window.open(arguments[0]);", car_link)
        driver.switch_to.window(driver.window_handles[-1])

        time.sleep(2)

        # PRICE
        price = wait.until(
            EC.visibility_of_element_located(
                (By.XPATH, "//p[contains(@class,'PriceSectionV3__ogPrice')]")
            )
        ).text

        # OVERVIEW
        overview_data = {}
        overview_section = wait.until(
            EC.presence_of_element_located((By.XPATH, "//section[@data-category='overview']"))
        )

        overview_items = overview_section.find_elements(
            By.XPATH, ".//div[contains(@class,'DesktopOverview__overviewItem')]"
        )

        for item in overview_items:
            key = item.find_element(By.XPATH, ".//div[contains(@class,'itemLabel')]").text
            value = item.find_element(By.XPATH, ".//div[contains(@class,'itemDisplay')]").text
            overview_data[key] = value

        # SPECIFICATIONS
        spec_data = {}
        spec_section = wait.until(
            EC.presence_of_element_located((By.XPATH, "//section[@data-category='features-specification']"))
        )

        spec_items = spec_section.find_elements(
            By.XPATH, ".//div[contains(@class,'styles__carSpecItem')]"
        )

        for spec in spec_items:
            key = spec.find_element(By.XPATH, ".//p").text
            value = spec.find_element(By.XPATH, ".//h4").text
            spec_data[key] = value

        # COMBINE
        car_data = {}
        car_data["Car Name"] = car_name
        car_data["Price"] = price
        car_data.update(overview_data)
        car_data.update(spec_data)

        master_data.append(car_data)

        driver.close()
        driver.switch_to.window(driver.window_handles[0])

        time.sleep(1)

    except Exception as e:
        print("Error:", e)
        driver.switch_to.window(driver.window_handles[0])
        continue

# ---------------- SAVE ----------------

df = pd.DataFrame(master_data)
df.to_csv("spinny_master_Delhi_NCR_dataset.csv", index=False)
df.to_excel("spinny_master_Delhi_NCR_dataset.xlsx", index=False)

print("\nDataset saved successfully")

driver.quit()


No popup
Cars loaded: 22
Cars loaded: 22
Cars loaded: 42
Cars loaded: 42
Cars loaded: 42
Cars loaded: 62
Cars loaded: 62
Cars loaded: 82
Cars loaded: 82
Cars loaded: 102
Cars loaded: 102
Cars loaded: 122
Cars loaded: 122
Cars loaded: 142
Cars loaded: 142
Cars loaded: 162
Cars loaded: 162
Cars loaded: 182
Cars loaded: 182
Cars loaded: 202
Cars loaded: 202
Cars loaded: 222
Cars loaded: 222
Cars loaded: 242
Cars loaded: 242
Cars loaded: 242
Cars loaded: 262
Cars loaded: 262
Cars loaded: 282
Cars loaded: 282
Cars loaded: 302
Cars loaded: 302
Cars loaded: 322
Cars loaded: 322
Cars loaded: 342
Cars loaded: 342
Cars loaded: 362
Cars loaded: 362
Cars loaded: 382
Cars loaded: 382
Cars loaded: 402
Cars loaded: 402
Cars loaded: 422
Cars loaded: 422
Cars loaded: 442
Cars loaded: 442
Cars loaded: 442
Cars loaded: 462
Cars loaded: 482
Cars loaded: 482
Cars loaded: 482
Cars loaded: 502
Cars loaded: 502
Cars loaded: 522
Cars loaded: 522
Cars loaded: 542
Cars loaded: 542
Cars loaded: 562
Cars loaded: 5